# 006 · Secure Randomness with secrets

> **Standard Library › Numbers & Math** · Python 3.14

The `secrets` module produces **cryptographically secure** random values from the operating system's secure source. Use it for anything an attacker must not be able to guess: session tokens, password-reset links, API keys, one-time codes, generated passwords and salts.

**In this notebook you will learn:**

- why `random` is unsafe for security and `secrets` is not
- generating tokens: `token_bytes`, `token_hex`, `token_urlsafe` — and how big they should be
- secure integers and choices: `randbelow`, `randbits`, `choice`
- constant-time comparison with `compare_digest`
- a complete password / one-time-code generator

## .NET mental model

| .NET (`System.Security.Cryptography`) | Python `secrets` |
|---|---|
| `RandomNumberGenerator.GetBytes(32)` | `secrets.token_bytes(32)` |
| `Convert.ToHexString(RandomNumberGenerator.GetBytes(32))` / `RandomNumberGenerator.GetHexString(64)` (.NET 8+) | `secrets.token_hex(32)` |
| Base64Url of random bytes (`Base64Url.EncodeToString`, .NET 9+) | `secrets.token_urlsafe(32)` |
| `RandomNumberGenerator.GetInt32(n)` | `secrets.randbelow(n)` |
| `RandomNumberGenerator.GetItems(choices, n)` (.NET 8+) | `secrets.choice(seq)` in a loop |
| `RandomNumberGenerator.GetString(choices, n)` (.NET 8+) | `"".join(secrets.choice(alphabet) for _ in range(n))` |
| `CryptographicOperations.FixedTimeEquals(a, b)` | `secrets.compare_digest(a, b)` |

Rule of thumb, same as in .NET: **`System.Random` ↔ `random`** for simulations; **`RandomNumberGenerator` ↔ `secrets`** for security.

## Why not `random`?

`random` uses the Mersenne Twister, a deterministic algorithm: given enough outputs (624 × 32 bits), its whole internal state — and therefore every future value — can be reconstructed. If you seed it, anyone who knows or guesses the seed can reproduce your "random" tokens *(see Standard Library › Numbers & Math › 005 · Pseudo-Random Numbers with random)*.

`secrets` reads from the OS cryptographic generator (`os.urandom`, which uses `getrandom()` on Linux, `BCryptGenRandom` on Windows, and the platform CSPRNG on macOS). It cannot be seeded — which is exactly the point.

⚠️ Because every value is different on every run, the cells below print **lengths and properties** rather than the values themselves.

## Tokens: `token_bytes`, `token_hex`, `token_urlsafe`

```csharp
byte[] key = RandomNumberGenerator.GetBytes(32);
string hex = Convert.ToHexString(RandomNumberGenerator.GetBytes(32)).ToLowerInvariant();
```

Each function takes the number of **random bytes** (`nbytes`); the text forms are longer than that because of encoding.

| Function | Returns | Length for `nbytes=32` |
|---|---|---|
| `token_bytes(n)` | `bytes` | 32 bytes |
| `token_hex(n)` | `str` of hex digits | 64 characters (2 per byte) |
| `token_urlsafe(n)` | `str`, Base64 URL-safe, no padding | ~43 characters (≈1.3 per byte) |

In [ ]:
import secrets

raw = secrets.token_bytes(32)
hex_token = secrets.token_hex(32)
url_token = secrets.token_urlsafe(32)

print(type(raw).__name__, len(raw))
print(type(hex_token).__name__, len(hex_token), all(c in "0123456789abcdef" for c in hex_token))
print(type(url_token).__name__, len(url_token))

A `token_hex(16)` looks like `"3f1c9a..."` (32 hex chars); a `token_urlsafe(16)` looks like `"q8Zr-..."` and uses only `A–Z a–z 0–9 - _`, so it can go straight into a URL or a cookie without escaping:

In [ ]:
import string

url_safe_chars = set(string.ascii_letters + string.digits + "-_")
token = secrets.token_urlsafe(16)
print(len(token), set(token) <= url_safe_chars)

reset_link = f"https://example.test/reset?token={secrets.token_urlsafe(32)}"
print(reset_link.startswith("https://example.test/reset?token="), len(reset_link))

### How many bytes?

If you call them with no argument, you get a sensible default (currently 32 bytes = 256 bits). 32 bytes is plenty for session IDs, reset links and API keys; 16 bytes (128 bits) is the usual minimum for anything guessable over the network.

In [ ]:
print(len(secrets.token_bytes()), len(secrets.token_hex()), len(secrets.token_urlsafe()))

## Secure integers and choices

| Function | Returns | .NET |
|---|---|---|
| `randbelow(n)` | int in `[0, n)` | `RandomNumberGenerator.GetInt32(n)` |
| `randbits(k)` | int with `k` random bits | — |
| `choice(seq)` | one element | `GetItems(seq, 1)[0]` |
| `SystemRandom()` | a full `random.Random`-compatible API on the secure source | — |

⚠️ **Gotcha:** `randbelow(n)` excludes `n`, like `GetInt32(n)` — unlike `random.randint`, whose upper bound is inclusive.

In [ ]:
dice = [secrets.randbelow(6) + 1 for _ in range(1000)]
print(min(dice), max(dice))                       # 1 6

six_digit_code = f"{secrets.randbelow(1_000_000):06d}"   # one-time code, keeps leading zeros
print(len(six_digit_code), six_digit_code.isdigit())

print(secrets.randbits(16) < 2**16)
print(secrets.choice(["rock", "paper", "scissors"]) in {"rock", "paper", "scissors"})

**Tip:** need a secure `sample` or `shuffle`? Use `secrets.SystemRandom()` — it has the whole `random.Random` API.

In [ ]:
secure_rng = secrets.SystemRandom()
winners = secure_rng.sample(["alice", "bob", "carol", "dave"], 2)
print(len(winners), len(set(winners)))           # 2 distinct winners

## Generating a password

```csharp
// .NET 8+
string password = RandomNumberGenerator.GetString(
    "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789!@#$%", 16);
```

Build the alphabet with the `string` module and pick each character with `secrets.choice`. To **enforce** character classes, generate until the password satisfies the policy — re-drawing keeps every valid password equally likely (don't "fix up" a fixed position, which reduces randomness).

In [ ]:
import string

ALPHABET = string.ascii_letters + string.digits + "!@#$%^&*-_"

def generate_password(length: int = 16) -> str:
    while True:
        password = "".join(secrets.choice(ALPHABET) for _ in range(length))
        if (any(c.islower() for c in password)
                and any(c.isupper() for c in password)
                and sum(c.isdigit() for c in password) >= 2
                and any(c in "!@#$%^&*-_" for c in password)):
            return password

pw = generate_password()
print(len(pw), any(c.isupper() for c in pw), sum(c.isdigit() for c in pw) >= 2)

A diceware-style passphrase is easier to type and remember. A real one uses a word list of several thousand words; here's the shape of it:

In [ ]:
WORDS = ["anchor", "basket", "candle", "dragon", "engine", "forest", "garden", "harbor",
         "island", "jungle", "kettle", "ladder", "meadow", "noodle", "orange", "pepper"]

passphrase = "-".join(secrets.choice(WORDS) for _ in range(5))
print(passphrase.count("-") + 1, "words")

⚠️ **Gotcha:** generated passwords are still secrets. Never log them, and never store them in plain text — hash them with a password-hashing function (e.g. `hashlib.scrypt`) *(see Standard Library › Data Formats › 006 · Encoding and Hashing with base64 and hashlib)*.

## `compare_digest` — constant-time comparison

When you check a token, signature or API key with `==`, the comparison stops at the **first differing character**. By timing many requests, an attacker can discover how many leading characters were right and recover the secret one character at a time — a *timing attack*.

```csharp
bool ok = CryptographicOperations.FixedTimeEquals(expectedBytes, providedBytes);
```

`secrets.compare_digest(a, b)` (the same function as `hmac.compare_digest`) takes the same time regardless of where the inputs differ.

In [ ]:
stored_token = "fake-token-0123456789abcdef"     # obviously fake example value

def is_valid(provided: str) -> bool:
    return secrets.compare_digest(provided, stored_token)

print(is_valid("fake-token-0123456789abcdef"))   # True
print(is_valid("fake-token-0000000000000000"))   # False

It accepts two `str` (ASCII only) or two bytes-like objects. Mixing types, or non-ASCII strings, raises `TypeError` — encode to bytes first:

In [ ]:
for a, b in [("abc", b"abc"), ("café", "café")]:
    try:
        secrets.compare_digest(a, b)
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")

print(secrets.compare_digest("café".encode(), "café".encode()))   # True

**Tip:** for comparing a password against a stored hash, compare the **hashes** with `compare_digest` (or use a library that does), never the plain text.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Random bytes | `RandomNumberGenerator.GetBytes(n)` | `secrets.token_bytes(n)` |
| Hex token | `RandomNumberGenerator.GetHexString(2n)` | `secrets.token_hex(n)` |
| URL-safe token | Base64Url of random bytes | `secrets.token_urlsafe(n)` |
| Int in [0, n) | `RandomNumberGenerator.GetInt32(n)` | `secrets.randbelow(n)` |
| Random element | `RandomNumberGenerator.GetItems(seq, 1)[0]` | `secrets.choice(seq)` |
| Password from alphabet | `RandomNumberGenerator.GetString(chars, n)` | `"".join(secrets.choice(chars) for _ in range(n))` |
| Secure sample/shuffle | — | `secrets.SystemRandom().sample(...)` |
| Constant-time equality | `CryptographicOperations.FixedTimeEquals` | `secrets.compare_digest(a, b)` |
| Simulations / games | `System.Random` | `random` — never for security |

## Exercises

1. Write `new_api_key(prefix: str) -> str` that returns keys like `"demo_key_<43 url-safe chars>"`. Check the length and prefix.
2. Write `otp(digits: int = 6) -> str` that returns a numeric one-time code with leading zeros preserved. Generate 1 000 codes and verify they all have the right length.
3. Write `verify_api_key(provided: str, expected: str) -> bool` safely, and show that it rejects a key that differs only in the last character.
4. Generate a 20-character password from letters and digits that contains at least 3 digits. Print its length and digit count (not the password).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import secrets
import string

# 1 — token_urlsafe(32) gives ~43 characters
def new_api_key(prefix: str) -> str:
    return f"{prefix}{secrets.token_urlsafe(32)}"

key = new_api_key("demo_key_")
print(key.startswith("demo_key_"), len(key))          # True 52

# 2 — randbelow excludes the upper bound; :0{n}d keeps leading zeros
def otp(digits: int = 6) -> str:
    return f"{secrets.randbelow(10 ** digits):0{digits}d}"

print(all(len(otp()) == 6 for _ in range(1000)))      # True

# 3 — constant-time comparison
def verify_api_key(provided: str, expected: str) -> bool:
    return secrets.compare_digest(provided.encode(), expected.encode())

expected = "demo_key_fake000000000000000000001"
print(verify_api_key(expected, expected))              # True
print(verify_api_key(expected[:-1] + "2", expected))   # False

# 4 — re-draw until the policy is satisfied
alphabet = string.ascii_letters + string.digits
while True:
    pw = "".join(secrets.choice(alphabet) for _ in range(20))
    if sum(c.isdigit() for c in pw) >= 3:
        break
print(len(pw), sum(c.isdigit() for c in pw) >= 3)      # 20 True
```

</details>